# CD Atlas: Compositional Analysis with scCODA

Bayesian compositional analysis of cell type abundance across CD conditions
(Control, Inactive, Active) using scCODA via pertpy. Analyses are run:

1. **Overall** – all cell categories by disease location × condition
2. **T cell subset** – T cell types by tissue × condition
3. **Myeloid subset** – myeloid types by tissue × condition
4. **Tissue-stratified** – colon and ileum separately (Control vs Inactive vs Active)

CD patients (condition == 'CD') and the P17 pooled-patient dataset are excluded
from the main analysis.

**Input**: `cleaned_annoed_all_cell_types.h5ad`

In [ ]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import pertpy as pt
import scvi

scvi.settings.dl_num_workers = 0

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
DATA_DIR   = "/path/to/cd/scrna/output"
MERGED_DIR = f"{DATA_DIR}/merged_cd"
OUTPUT_DIR = f"{DATA_DIR}/sccoda"

## 1. Load atlas and compute summary metadata

In [ ]:
adata = sc.read_h5ad(f"{MERGED_DIR}/cleaned_annoed_all_cell_types.h5ad")

# Dataset identifier from sample prefix
adata.obs['dataset'] = [i.split('_')[0] for i in adata.obs['sample'].tolist()]

# Harmonize P17 patients (cells pooled across 2-3 patients per run)
patient_new = []
for _, row in adata.obs.iterrows():
    if row['dataset'] == 'P17':
        patient_new.append('P17_1-2' if row['sample'] in ['P17_1','P17_2','P17_3','P17_4']
                           else 'P17_3-5')
    else:
        patient_new.append(row['patient'])
adata.obs['patient'] = patient_new

# Condition summary: map all active variants → Active
adata.obs['condition_summary'] = adata.obs['condition'].map({
    'Control':              'Control',
    'Inactive':             'Inactive',
    'Inactive (Trt Naive)': 'Inactive',
    'Active':               'Active',
    'Active_mild':          'Active',
    'Active_moderate':      'Active',
    'Active_severe':        'Active',
    'Active (Refractory)':  'Active',
    'Active (Trt Naive)':   'Active',
    'CD':                   'CD',
}).fillna('Other')

# Tissue summary
def tissue_bin(tr):
    if any(x in str(tr) for x in ['Ileum','Duodenum','Jejunum','Small Bowel']):
        return 'Ileum'
    return 'Colon'

adata.obs['tissue_summary'] = adata.obs['tissue_region'].apply(tissue_bin)

# Exclude CD patients (Crohn's classification separate from IBD severity) and P17
adata_nocd      = adata[~adata.obs['condition_summary'].isin(['CD', 'Other'])]
adata_nocd_no17 = adata_nocd[adata_nocd.obs['dataset'] != 'P17']

print(adata_nocd_no17.obs['condition_summary'].value_counts())

## 2. Overall compositional analysis by disease location × condition

In [ ]:
# Combined covariate: condition_summary × tissue_summary
adata_nocd_no17.obs['tissue_cond'] = (
    adata_nocd_no17.obs['condition_summary'] + '_' +
    adata_nocd_no17.obs['tissue_summary']
)
adata_nocd_no17.obs['tissue_cond'] = pd.Categorical(
    adata_nocd_no17.obs['tissue_cond'],
    categories=['Control_Colon','Control_Ileum',
                'Inactive_Colon','Inactive_Ileum',
                'Active_Colon','Active_Ileum'],
    ordered=True
)
print(adata_nocd_no17.obs['tissue_cond'].value_counts())

In [ ]:
scc = pt.tl.Sccoda()

mdata_tis = scc.load(
    adata_nocd_no17,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell category",
    sample_identifier="sample",
    covariate_obs=["tissue_cond"]
)
mdata_tis = scc.prepare(
    mdata_tis,
    formula="C(tissue_cond)",
    reference_cell_type="Endothelial"
)
scc.run_nuts(mdata_tis)

In [ ]:
scc.summary(mdata_tis, extended=True)
eff_tis  = scc.get_effect_df(mdata_tis)
cred_tis = scc.credible_effects(mdata_tis)
eff_tis.to_csv(f"{OUTPUT_DIR}/sccoda_overall_tissue_cond_effects.csv")

## 3. T cell compositional analysis (ileum)

In [ ]:
adata_t_il = adata_nocd_no17[
    (adata_nocd_no17.obs['cell category'] == 'T') &
    (adata_nocd_no17.obs['tissue_summary'] == 'Ileum')
]

scc_t = pt.tl.Sccoda()
mdata_t_il = scc_t.load(
    adata_t_il,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell type",
    sample_identifier="sample",
    covariate_obs=["condition_summary"]
)
mdata_t_il = scc_t.prepare(
    mdata_t_il,
    formula="C(condition_summary)",
    reference_cell_type="automatic",
    automatic_reference_absence_threshold=0.2
)
scc_t.run_nuts(mdata_t_il)

In [ ]:
scc_t.summary(mdata_t_il, extended=True)
eff_t_il  = scc_t.get_effect_df(mdata_t_il)
cred_t_il = scc_t.credible_effects(mdata_t_il)
eff_t_il.to_csv(f"{OUTPUT_DIR}/sccoda_tcell_ileum_effects.csv")

## 4. T cell compositional analysis (colon)

In [ ]:
adata_t_col = adata_nocd_no17[
    (adata_nocd_no17.obs['cell category'] == 'T') &
    (adata_nocd_no17.obs['tissue_summary'] == 'Colon')
]

scc_tc = pt.tl.Sccoda()
mdata_t_col = scc_tc.load(
    adata_t_col,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell type",
    sample_identifier="sample",
    covariate_obs=["condition_summary"]
)
mdata_t_col = scc_tc.prepare(
    mdata_t_col,
    formula="C(condition_summary)",
    reference_cell_type="automatic",
    automatic_reference_absence_threshold=0.2
)
scc_tc.run_nuts(mdata_t_col)

In [ ]:
scc_tc.summary(mdata_t_col, extended=True)
eff_t_col  = scc_tc.get_effect_df(mdata_t_col)
cred_t_col = scc_tc.credible_effects(mdata_t_col)
eff_t_col.to_csv(f"{OUTPUT_DIR}/sccoda_tcell_colon_effects.csv")

## 5. Forest plot: T cell scCODA results (Ileum + Colon)

In [ ]:
import re

def tidy_sccoda(df, tissue_name):
    df = df.copy()
    df['Covariate'] = df['Covariate'].ffill()
    cond = df['Covariate'].str.extract(
        r'C\(condition_summary\)T\.([^_]+)', expand=False
    )
    df['Condition'] = cond
    df = df.rename(columns={
        'Final Parameter': 'beta',
        'HDI 3%':  'hdi_low',
        'HDI 97%': 'hdi_high'
    })
    df['Tissue'] = tissue_name
    df = df[df['Condition'].isin(['Inactive', 'Active'])]
    for c in ['beta', 'hdi_low', 'hdi_high']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    return df[['Tissue', 'Condition', 'Cell Type', 'beta', 'hdi_low', 'hdi_high']]

ileum_tidy = tidy_sccoda(eff_t_il,  'Ileum')
colon_tidy = tidy_sccoda(eff_t_col, 'Colon')
df_all     = pd.concat([ileum_tidy, colon_tidy], ignore_index=True)
df_all.to_csv(f"{OUTPUT_DIR}/sccoda_tcell_combined_effects.csv", index=False)

In [ ]:
cell_order = sorted(df_all['Cell Type'].unique().tolist())
group_gap  = 1.6
x_base     = np.arange(len(cell_order)) * group_gap

pairs   = [('Colon','Inactive'), ('Ileum','Inactive'), ('Colon','Active'), ('Ileum','Active')]
offsets = {pairs[i]: within for i, within in enumerate([-0.3, -0.1, 0.1, 0.3])}
colors  = {
    ('Colon','Inactive'): '#86efac',
    ('Ileum','Inactive'): '#16a34a',
    ('Colon','Active'):   '#fdba74',
    ('Ileum','Active'):   '#f97316',
}

fig, ax = plt.subplots(figsize=(7, 5))
for (tissue, cond) in pairs:
    sub = (df_all[(df_all['Tissue']==tissue) & (df_all['Condition']==cond)]
           .set_index('Cell Type').reindex(cell_order).dropna(subset=['beta','hdi_low','hdi_high']))
    idx  = [cell_order.index(ct) for ct in sub.index]
    y    = sub['beta'].values
    yerr = np.vstack([y - sub['hdi_low'].values, sub['hdi_high'].values - y])
    ax.errorbar(
        x=x_base[np.array(idx)] + offsets[(tissue, cond)], y=y, yerr=yerr,
        fmt='o', mfc=colors[(tissue,cond)], mec='white',
        ecolor=colors[(tissue,cond)], elinewidth=1.4, capsize=3, markersize=5
    )

ax.axhline(0, color='#666', lw=1, ls='--')
ax.set_xticks(x_base)
ax.set_xticklabels(cell_order, rotation=30, ha='right')
ax.set_ylabel('Posterior Mean (3–97% HDI)')
ax.set_title('CD T Cell Composition: scCODA Results')

handles = [Line2D([0],[0], marker='o', color='none',
                  label=f"{t}_{c}", markerfacecolor=colors[(t,c)],
                  markeredgecolor='white', markersize=4)
           for (t,c) in pairs]
ax.legend(handles=handles, frameon=False, ncol=2, fontsize=9.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/sccoda_tcell_forest_plot.pdf", bbox_inches='tight', dpi=300)
plt.show()

## 6. Myeloid compositional analysis (ileum)

In [ ]:
adata_mye_il = adata_nocd_no17[
    (adata_nocd_no17.obs['cell category'] == 'Myeloid') &
    (adata_nocd_no17.obs['tissue_summary'] == 'Ileum')
]

scc_m = pt.tl.Sccoda()
mdata_mye_il = scc_m.load(
    adata_mye_il,
    type="cell_level",
    generate_sample_level=True,
    cell_type_identifier="cell type",
    sample_identifier="sample",
    covariate_obs=["condition_summary"]
)
mdata_mye_il = scc_m.prepare(
    mdata_mye_il,
    formula="C(condition_summary)",
    reference_cell_type="automatic",
    automatic_reference_absence_threshold=0.2
)
scc_m.run_nuts(mdata_mye_il)

scc_m.summary(mdata_mye_il, extended=True)
eff_mye_il = scc_m.get_effect_df(mdata_mye_il)
eff_mye_il.to_csv(f"{OUTPUT_DIR}/sccoda_myeloid_ileum_effects.csv")